# MSDS 565 — Unit 1, Notebook 3: Modeling 
**Team DataStars**

Train, evaluate, and compare regressors on your preprocessed dataset.
If your R² is suspiciously close to 1, you have likely leaked price information into a
feature — double check your Notebook 2 feature engineering.

## 1.1 Setup and Imports

Import the libraries needed for data handling, model training,
evaluation, model saving, and neural networks.

In [34]:
# Core libraries
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Scikit-learn models and utilities
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

# Model saving
import joblib

# Keras / TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 1.2 Load Processed Training and Test Data

Notebook 2 created and exported an 80/20 train/test split as
`processed_train.csv` and `processed_test.csv`. These same datasets are loaded
here so that all models are trained and evaluated using the same observations.

In [2]:
# Load the processed train/test split created in Notebook 2

train_df = pd.read_csv("../../data/processed_train.csv")
test_df = pd.read_csv("../../data/processed_test.csv")

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (237268, 312)
Test data shape: (59317, 312)


## 1.3 Define the Target

The regression target is `price`.

The target is separated from the predictor variables before model training.
Because `price` is the value the models are attempting to predict, it must not
be included among the predictor features.

In [3]:
# Define the regression target

y_train = train_df["price"].copy()
y_test = test_df["price"].copy()

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

y_train shape: (237268,)
y_test shape: (59317,)


### 1.4 Identify Metadata Columns

The original categorical metadata columns use the `object` data type because
they contain text values. These original columns are preserved for later
subgroup and fairness analysis but are not used directly for modeling.

Notebook 2 created numeric one-hot-encoded versions of categorical features,
such as `make_name_Toyota` and `make_name_Ford`. Those numeric encoded
features can be used by the models.

In [4]:
# Identify object-dtype metadata columns.

metadata_cols = train_df.select_dtypes(include="object").columns.tolist()

print(f"Number of metadata columns: {len(metadata_cols)}")
print("\nMetadata columns:")
print(metadata_cols)

Number of metadata columns: 18

Metadata columns:
['city', 'bed', 'body_type', 'cabin', 'engine_type', 'exterior_color', 'franchise_make', 'fuel_type', 'interior_color', 'listing_color', 'major_options', 'make_name', 'model_name', 'sp_name', 'transmission', 'transmission_display', 'trim_name', 'wheel_system']


## 1.5 Construct Modeling Feature Matrices

The modeling feature matrices are created by removing the target (`price`)
and all object-dtype metadata columns.

The metadata columns remain in the original processed datasets for later
subgroup fairness analysis, while only numeric model-ready features are
included in `X_train` and `X_test`.

In [5]:
# Construct model-ready feature matrices

X_train = train_df.drop(
    columns=["price"] + metadata_cols
).copy()

X_test = test_df.drop(
    columns=["price"] + metadata_cols
).copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (237268, 293)
X_test shape: (59317, 293)


In [6]:
# Confirm train and test contain identical modeling features

assert list(X_train.columns) == list(X_test.columns), \
    "Train and test feature columns do not match."

print("Train/test feature columns match.")

Train/test feature columns match.


## 1.6 Modeling Data Quality Check

Before model training, the feature matrices are checked to confirm that:

- `price` is not included as a predictor,
- no object-dtype metadata columns remain,
- the training and test feature sets contain the same columns,
- no missing values remain, and
- no infinite values remain.

These checks help prevent target leakage and modeling errors.

In [7]:
print("Price in X_train:", "price" in X_train.columns)
print("Price in X_test:", "price" in X_test.columns)

print(
    "Object columns in X_train:",
    X_train.select_dtypes(include="object").columns.tolist()
)

print(
    "Object columns in X_test:",
    X_test.select_dtypes(include="object").columns.tolist()
)

print(
    "Missing values in X_train:",
    X_train.isna().sum().sum()
)

print(
    "Missing values in X_test:",
    X_test.isna().sum().sum()
)

print(
    "Infinite values in X_train:",
    np.isinf(X_train.select_dtypes(include="number")).sum().sum()
)

print(
    "Infinite values in X_test:",
    np.isinf(X_test.select_dtypes(include="number")).sum().sum()
)

Price in X_train: False
Price in X_test: False
Object columns in X_train: []
Object columns in X_test: []
Missing values in X_train: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0


In [8]:
# Inspect the final modeling features

print("Number of modeling features:", X_train.shape[1])

print("\nModeling data types:")
display(X_train.dtypes.value_counts())

print("\nFirst 20 modeling features:")
print(X_train.columns[:20].tolist())

Number of modeling features: 293

Modeling data types:


float64    293
Name: count, dtype: int64


First 20 modeling features:
['back_legroom', 'bed_length', 'city_fuel_economy', 'daysonmarket', 'engine_displacement', 'fleet', 'frame_damaged', 'franchise_dealer', 'front_legroom', 'fuel_tank_volume', 'has_accidents', 'height', 'highway_fuel_economy', 'horsepower', 'isCab', 'is_cpo', 'is_new', 'is_oemcpo', 'latitude', 'length']


This code confirms that the metadata was included for modeling.

# 2. Model Evaluation Strategy

## 2.1 Evaluation Metrics

Model performance will be evaluated using three regression metrics:

- **Mean Absolute Error (MAE):** Measures the average absolute difference
  between the predicted and actual vehicle prices. Lower values indicate
  better performance.

- **Mean Absolute Percentage Error (MAPE):** Measures the average prediction
  error as a percentage of the actual vehicle price. Lower values indicate
  better performance.

- **R² (Coefficient of Determination):** Measures how much of the variation
  in vehicle price is explained by the model. Values closer to 1 indicate
  better predictive performance.

Using multiple metrics provides a more complete view of model performance
than relying on a single measure.

## 2.2 Five-Fold Cross-Validation

Five-fold cross-validation will be used to evaluate model performance during
training.

The training dataset is divided into five folds. Each model is trained on
four folds and validated on the remaining fold. This process is repeated
five times so that each fold serves as the validation set once.

The held-out test dataset is not used during cross-validation or
hyperparameter selection. It is reserved for final model evaluation.

In [9]:
# Define 5-fold cross-validation

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

KFold(n_splits=5, random_state=42, shuffle=True)


We used five-fold cross-validation to randomize the training observations before creating the folds and random state of 42 to ensure the folds are reproducible. To double check before running the model we checked the target. 

In [10]:
# Inspect the target variable before model training

print("Training price range:")
print(f"Minimum: ${y_train.min():,.2f}")
print(f"Maximum: ${y_train.max():,.2f}")

print("\nTest price range:")
print(f"Minimum: ${y_test.min():,.2f}")
print(f"Maximum: ${y_test.max():,.2f}")

print("\nZero prices in training:", (y_train == 0).sum())
print("Zero prices in test:", (y_test == 0).sum())

Training price range:
Minimum: $794.00
Maximum: $1,750,000.00

Test price range:
Minimum: $670.00
Maximum: $599,900.00

Zero prices in training: 0
Zero prices in test: 0


# 3. Scikit-Learn Regression Models

Three scikit-learn regression models will be trained and compared.

Each model will use 5-fold `GridSearchCV` for hyperparameter selection.
Performance will be evaluated using cross-validation MAE, MAPE, and R²,
followed by evaluation on the held-out test dataset.

Training time will also be recorded for comparison.

### 3.1.1 Linear Regression

Linear Regression is used as the baseline regression model.

The model estimates vehicle price as a linear combination of the predictor
features. A small hyperparameter grid is evaluated using 5-fold
`GridSearchCV`.

The best model selected through cross-validation will then be evaluated on
the held-out test dataset.

In [11]:
# Define Linear Regression model

linear_model = LinearRegression()

# Hyperparameter grid
linear_param_grid = {
    "fit_intercept": [True, False],
    "positive": [False, True]
}

### 3.1.2 Define the scoring metrics

In [12]:
# Scoring metrics for cross-validation

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MAPE": "neg_mean_absolute_percentage_error",
    "R2": "r2"
}

### 3.1.3 Create GridSearchCV

In [ ]:
# Set up GridSearchCV for Linear Regression

linear_grid = GridSearchCV(
    estimator=linear_model,
    param_grid=linear_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

linear_grid

In [ ]:
# Train and tune Linear Regression

linear_grid.fit(X_train, y_train)

# Get the training time for the final best-estimator refit
linear_train_time = linear_grid.refit_time_

print("Best parameters:", linear_grid.best_params_)
print(f"Final refit training time: {linear_train_time:.2f} seconds")

Of the parameter combinations tested, choose the one with the best MAE, then after finding the best parameter combination GridSearchCV automatically refits the winning configuration on the entire training dataset.

### 3.1.4 Linear Regression Cross-Validation Results

The best Linear Regression hyperparameter combination was selected using
5-fold cross-validation with MAE as the refit metric.

The mean cross-validation MAE, MAPE, and R² for the selected model are
reported below. MAE and MAPE are converted back to positive values because
scikit-learn stores error-based scoring metrics as negative values during
model selection.

In [15]:
# Get the index of the best parameter combination
best_index = linear_grid.best_index_

# Extract cross-validation scores for the winning model
linear_cv_mae = -linear_grid.cv_results_[
    "mean_test_MAE"
][best_index]

linear_cv_mape = -linear_grid.cv_results_[
    "mean_test_MAPE"
][best_index]

linear_cv_r2 = linear_grid.cv_results_[
    "mean_test_R2"
][best_index]

print("Linear Regression - 5-Fold CV Results")
print(f"MAE:  ${linear_cv_mae:,.2f}")
print(f"MAPE: {linear_cv_mape * 100:.2f}%")
print(f"R²:   {linear_cv_r2:.4f}")

Linear Regression - 5-Fold CV Results
MAE:  $5,554.99
MAPE: 26.75%
R²:   0.6801


### Linear Regression CV Interpretation:  
- MAE **$5,554.99** LR predictions were off by an average of $5555 either high or low.
- MAPE **26.75%** predictions differed from actual prices by about 26.75% on average.
- R2 **0.6801** LR model explains aobut 68% of the variation in model price.
- These results provide a baseline for comparison with the more complex regression models.

### 3.1.5 Evaluate Linear Regression on Held-Out Test Set

In [16]:
# Get the best Linear Regression model
best_linear = linear_grid.best_estimator_

# Generate predictions on the held-out test set
linear_test_pred = best_linear.predict(X_test)

# Calculate test metrics
linear_test_mae = mean_absolute_error(
    y_test,
    linear_test_pred
)

linear_test_mape = mean_absolute_percentage_error(
    y_test,
    linear_test_pred
)

linear_test_r2 = r2_score(
    y_test,
    linear_test_pred
)

print("Linear Regression - Test Results")
print(f"MAE:  ${linear_test_mae:,.2f}")
print(f"MAPE: {linear_test_mape * 100:.2f}%")
print(f"R²:   {linear_test_r2:.4f}")

Linear Regression - Test Results
MAE:  $5,498.22
MAPE: 26.86%
R²:   0.7339


### Linear Regression Test Interpretation

On the held-out test set, Linear Regression achieved an MAE of approximately
**$5,498**, a MAPE of **26.86%**, and an R² of **0.7339**.

The test results were reasonably consistent with the 5-fold cross-validation
results. Test MAE was slightly lower than cross-validation MAE, MAPE was nearly
identical, and test R² was somewhat higher. This suggests that the Linear
Regression model generalized reasonably well to unseen data.

These results will serve as the baseline for comparison with the more complex
regression models.

### 3.1.6 Record Linear Regression Results

In [17]:
# Store Linear Regression results

model_results = []

model_results.append({
    "Model": "Linear Regression",
    "Best Parameters": str(linear_grid.best_params_),
    "CV MAE": linear_cv_mae,
    "CV MAPE": linear_cv_mape,
    "CV R2": linear_cv_r2,
    "Test MAE": linear_test_mae,
    "Test MAPE": linear_test_mape,
    "Test R2": linear_test_r2,
    "Training Time (sec)": linear_train_time
})

results_df = pd.DataFrame(model_results)

display(results_df)

,Model,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
0,Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.2234,0.268621,0.733916,78.251818


### 3.1.7 Save Linear Regression Model

In [21]:
# Save the best Linear Regression model

joblib.dump(
    best_linear,
    "../../models/P1_N3_linreg.pkl"
)

print("Linear Regression model saved.")

Linear Regression model saved.


## 3.2 Random Forest Regression

Random Forest Regression is an ensemble learning method that combines
predictions from multiple decision trees.

Unlike Linear Regression, Random Forest can model nonlinear relationships
and interactions among vehicle characteristics without requiring those
relationships to be specified in advance.

A meaningful hyperparameter grid will be evaluated using 5-fold
GridSearchCV. The best model will then be evaluated on the held-out
test dataset.

### 3.2.1 Define the Random Forest

In [22]:
random_forest = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

### 3.2.2 Create the parameter grid

In [23]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.5]
}

### 3.2.3 Set up GridSearchCV

In [24]:
rf_grid = GridSearchCV(
    estimator=random_forest,
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

rf_grid

,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 20], 'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 2], 'n_estimators': [100, 200]}"
,scoring,"{'MAE': 'neg_mean_absolute_error', 'MAPE': 'neg_mean_absolute_percentage_error', 'R2': 'r2'}"
,n_jobs,-1
,refit,'MAE'
,cv,KFold(n_split... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [26]:
rf_grid.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid.best_params_)


Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters:
{'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}


**Interpretation:**  
GridSearchCV selected a Random Forest with 200 trees, unrestricted tree
depth, a minimum of one observation per leaf, and 50% of the available
features considered at each split. This combination produced the best
cross-validated MAE among the 16 hyperparameter combinations evaluated.

### 3.2.4 Random Forest Cross-Validation Results

The best Random Forest hyperparameter combination was selected using
5-fold cross-validation. The mean MAE, MAPE, and R² scores across the
five validation folds are extracted below to evaluate the model's
cross-validation performance.

In [27]:
# Extract cross-validation results for the best Random Forest model

best_rf_index = rf_grid.best_index_

rf_cv_mae = -rf_grid.cv_results_["mean_test_MAE"][best_rf_index]
rf_cv_mape = -rf_grid.cv_results_["mean_test_MAPE"][best_rf_index]
rf_cv_r2 = rf_grid.cv_results_["mean_test_R2"][best_rf_index]

print(f"Random Forest CV MAE: ${rf_cv_mae:,.2f}")
print(f"Random Forest CV MAPE: {rf_cv_mape:.2%}")
print(f"Random Forest CV R²: {rf_cv_r2:.4f}")

Random Forest CV MAE: $1,739.88
Random Forest CV MAPE: 7.18%
Random Forest CV R²: 0.8979


**Interpretation:**  
The Random Forest achieved a mean cross-validation MAE of $1,739.88,
a MAPE of 7.18%, and an R² of 0.8979. This represents a substantial
improvement over the Linear Regression baseline, which had a CV MAE
of $5,554.99, MAPE of 26.75%, and R² of 0.6801.

The improvement suggests that relationships between vehicle
characteristics and price are not purely linear. Random Forest can
capture nonlinear relationships and interactions among predictors
that Linear Regression cannot model as easily.

### 3.2.5 Evaluate Random Forest on the Held-Out Test Set

The best Random Forest model identified by GridSearchCV is evaluated
on the held-out test set. These observations were not used during
model training or cross-validation.

MAE, MAPE, and R² are calculated to measure how well the model
generalizes to unseen data.

In [28]:
# Get the best Random Forest model

best_rf = rf_grid.best_estimator_

# Make predictions on the held-out test set

rf_test_pred = best_rf.predict(X_test)

# Calculate test-set metrics

rf_test_mae = mean_absolute_error(y_test, rf_test_pred)

rf_test_mape = mean_absolute_percentage_error(
    y_test,
    rf_test_pred
)

rf_test_r2 = r2_score(y_test, rf_test_pred)

# Display results

print(f"Random Forest Test MAE: ${rf_test_mae:,.2f}")
print(f"Random Forest Test MAPE: {rf_test_mape:.2%}")
print(f"Random Forest Test R²: {rf_test_r2:.4f}")

Random Forest Test MAE: $1,638.97
Random Forest Test MAPE: 6.92%
Random Forest Test R²: 0.9503


**Interpretation:**  
The Random Forest achieved a test MAE of $1,638.97, a MAPE of 6.92%,
and an R² of 0.9503. Test performance was slightly better than the
mean cross-validation performance, indicating that the model
generalized well to the held-out data.

Compared with Linear Regression, the Random Forest produced
substantially lower prediction errors and a higher R², suggesting
that nonlinear relationships and interactions among the vehicle
features are important for predicting price.

Because the test R² is high, potential target leakage should continue
to be monitored as additional models are evaluated.

### 3.2.6 Record Random Forest Results

The Random Forest cross-validation metrics, held-out test metrics, and
final refit training time are recorded for comparison with the other
regression models.

The refit training time represents the time required for GridSearchCV
to retrain the best Random Forest configuration on the complete
training dataset.

In [29]:
# Get final Random Forest refit training time

rf_train_time = rf_grid.refit_time_

# Record Random Forest results

model_results.append({
    "Model": "Random Forest",
    "Best Parameters": str(rf_grid.best_params_),
    "CV MAE": rf_cv_mae,
    "CV MAPE": rf_cv_mape,
    "CV R2": rf_cv_r2,
    "Test MAE": rf_test_mae,
    "Test MAPE": rf_test_mape,
    "Test R2": rf_test_r2,
    "Training Time (sec)": rf_train_time
})

# Update the performance table

results_df = pd.DataFrame(model_results)

display(results_df)

,Model,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
0,Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,78.251818
1,Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554


### 3.2.7 Save the Best Random Forest Model

The best Random Forest model identified by GridSearchCV is saved for
later use. The saved model is the best estimator refitted on the
complete training dataset.

In [30]:
# Save the best Random Forest model

joblib.dump(
    best_rf,
    "../../models/P1_N3_rf.pkl"
)

print("Random Forest model saved.")

Random Forest model saved.


## 3.3 Ridge Regression

Ridge Regression is used as the third scikit-learn regression model.
Like Linear Regression, Ridge models linear relationships between the
predictors and vehicle price. However, Ridge adds L2 regularization,
which penalizes large coefficient values.

This can improve model stability when the dataset contains many
predictors or correlated features. GridSearchCV is used to determine
the best regularization strength using 5-fold cross-validation.

### 3.3.1 Define the Ridge Regression Model

A Ridge Regression model is created. Ridge Regression extends ordinary
Linear Regression by applying an L2 penalty to the model coefficients.
The strength of this penalty will be selected using GridSearchCV.

In [35]:
ridge_model = Ridge()

### 3.3.2 Create the Parameter Grid

A range of alpha values is evaluated to determine the appropriate
strength of L2 regularization. Smaller alpha values apply less
regularization, while larger values apply a stronger penalty to the
model coefficients.

In [36]:
ridge_param_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

### 3.3.3 Set Up GridSearchCV

GridSearchCV is used to evaluate each Ridge alpha value using the same
5-fold cross-validation strategy and evaluation metrics used for the
previous scikit-learn models. MAE is used to select the best model.

In [37]:
ridge_grid = GridSearchCV(
    estimator=ridge_model,
    param_grid=ridge_param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    return_train_score=False
)

ridge_grid

,estimator,Ridge()
,param_grid,"{'alpha': [0.01, 0.1, ...]}"
,scoring,"{'MAE': 'neg_mean_absolute_error', 'MAPE': 'neg_mean_absolute_percentage_error', 'R2': 'r2'}"
,n_jobs,-1
,refit,'MAE'
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.0


### 3.3.4 Fit the Ridge Grid Search

The Ridge models are trained using 5-fold cross-validation. GridSearchCV
selects the alpha value with the best mean validation MAE and refits
that configuration using the complete training dataset.

In [38]:
ridge_grid.fit(X_train, y_train)

print("Best parameters:")
print(ridge_grid.best_params_)

Best parameters:
{'alpha': 100.0}


**Interpretation:**  
GridSearchCV selected an alpha of 100.0 for Ridge Regression. This was
the strongest regularization value evaluated in the parameter grid and
produced the best mean cross-validation MAE among the candidate values.

### 3.3.5 Ridge Regression Cross-Validation Results

The cross-validation performance of the best Ridge Regression model is
evaluated using mean MAE, MAPE, and R² across the five validation folds.
These metrics allow Ridge Regression to be compared directly with the
other scikit-learn regression models.

In [39]:
# Extract cross-validation results for the best Ridge model

best_ridge_index = ridge_grid.best_index_

ridge_cv_mae = -ridge_grid.cv_results_["mean_test_MAE"][best_ridge_index]
ridge_cv_mape = -ridge_grid.cv_results_["mean_test_MAPE"][best_ridge_index]
ridge_cv_r2 = ridge_grid.cv_results_["mean_test_R2"][best_ridge_index]

print(f"Ridge Regression CV MAE: ${ridge_cv_mae:,.2f}")
print(f"Ridge Regression CV MAPE: {ridge_cv_mape:.2%}")
print(f"Ridge Regression CV R²: {ridge_cv_r2:.4f}")

Ridge Regression CV MAE: $5,550.67
Ridge Regression CV MAPE: 26.72%
Ridge Regression CV R²: 0.6802


**Interpretation:**  
Ridge Regression achieved a mean cross-validation MAE of $5,550.67,
a MAPE of 26.72%, and an R² of 0.6802. These results are only slightly
better than the Linear Regression baseline.

Although the selected alpha of 100.0 applied relatively strong
regularization, Ridge Regression did not substantially improve
predictive performance. This suggests that regularization alone does
not address the limitations of the linear model for this dataset.

Random Forest continues to perform substantially better, suggesting
that nonlinear relationships and interactions among vehicle
characteristics are important for predicting price.

### 3.3.6 Evaluate Ridge Regression on the Held-Out Test Set

The best Ridge Regression model selected through 5-fold
cross-validation is evaluated on the held-out test set. MAE, MAPE,
and R² are calculated to measure its ability to generalize to
unseen observations.

In [40]:
# Get the best Ridge Regression model

best_ridge = ridge_grid.best_estimator_

# Make predictions on the held-out test set

ridge_test_pred = best_ridge.predict(X_test)

# Calculate test-set metrics

ridge_test_mae = mean_absolute_error(y_test, ridge_test_pred)

ridge_test_mape = mean_absolute_percentage_error(
    y_test,
    ridge_test_pred
)

ridge_test_r2 = r2_score(y_test, ridge_test_pred)

# Display results

print(f"Ridge Regression Test MAE: ${ridge_test_mae:,.2f}")
print(f"Ridge Regression Test MAPE: {ridge_test_mape:.2%}")
print(f"Ridge Regression Test R²: {ridge_test_r2:.4f}")

Ridge Regression Test MAE: $5,494.76
Ridge Regression Test MAPE: 26.84%
Ridge Regression Test R²: 0.7340


**Interpretation:**  
Ridge Regression achieved a test MAE of $5,494.76, a MAPE of 26.84%,
and an R² of 0.7340. The test results are reasonably consistent with
the cross-validation results.

Ridge Regression performed only slightly better than ordinary Linear
Regression on both cross-validation and held-out test data. This
suggests that L2 regularization provides little additional predictive
benefit for the linear model in this dataset.

Random Forest remains substantially stronger than both linear models,
supporting the importance of nonlinear relationships and interactions
for predicting vehicle price.

### 3.3.7 Record Ridge Regression Results

The Ridge Regression cross-validation metrics, held-out test metrics,
and final refit training time are recorded for comparison with the
other regression models.

In [41]:
# Get final Ridge Regression refit training time

ridge_train_time = ridge_grid.refit_time_

# Record Ridge Regression results

model_results.append({
    "Model": "Ridge Regression",
    "Best Parameters": str(ridge_grid.best_params_),
    "CV MAE": ridge_cv_mae,
    "CV MAPE": ridge_cv_mape,
    "CV R2": ridge_cv_r2,
    "Test MAE": ridge_test_mae,
    "Test MAPE": ridge_test_mape,
    "Test R2": ridge_test_r2,
    "Training Time (sec)": ridge_train_time
})

# Update the performance table

results_df = pd.DataFrame(model_results)

display(results_df)

,Model,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
0,Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,78.251818
1,Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
2,Ridge Regression,{'alpha': 100.0},5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217


### 3.3.8 Save the Best Ridge Regression Model

The best Ridge Regression model identified by GridSearchCV is saved
for later use. The saved model is the best estimator refitted on the
complete training dataset.

In [42]:
# Save the best Ridge Regression model

joblib.dump(
    best_ridge,
    "../../models/P1_N3_ridge.pkl"
)

print("Ridge Regression model saved.")

Ridge Regression model saved.


## 4. Keras Neural Network Models

Three different neural network architectures are evaluated using Keras.
Each architecture is evaluated using manual 5-fold cross-validation on
the training data.

For each architecture, MAE, MAPE, and R² are calculated for each fold
and averaged across the five folds. The architecture with the best
cross-validation performance will then be retrained on the complete
training dataset and evaluated on the held-out test set.

### 4.1 Prepare Data for Keras

The same training and test datasets used for the scikit-learn models
are converted to NumPy arrays for use with Keras. No new train/test
split is created, preserving the original held-out test set.

In [44]:
# Convert modeling data to NumPy arrays for Keras

X_train_nn = X_train.to_numpy(dtype="float32")
X_test_nn = X_test.to_numpy(dtype="float32")

y_train_nn = y_train.to_numpy(dtype="float32")
y_test_nn = y_test.to_numpy(dtype="float32")

print("X_train_nn shape:", X_train_nn.shape)
print("X_test_nn shape:", X_test_nn.shape)
print("y_train_nn shape:", y_train_nn.shape)
print("y_test_nn shape:", y_test_nn.shape)

X_train_nn shape: (237268, 293)
X_test_nn shape: (59317, 293)
y_train_nn shape: (237268,)
y_test_nn shape: (59317,)


### 4.2 Manual 5-Fold Cross-Validation Strategy

Unlike the scikit-learn models, the Keras neural networks are evaluated
using a manual 5-fold cross-validation loop. For each fold, a new neural
network is created and trained using four folds, while the remaining
fold is used for validation.

A new model must be created for every fold so that information learned
during one fold does not carry over into another.

### 4.3 Keras Architecture 1

The first neural network serves as a relatively simple baseline
architecture. It contains two hidden Dense layers using ReLU activation
and a single output neuron for predicting vehicle price.

In [46]:
def build_nn1(input_dim):
    
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )
    
    return model

In [47]:
nn1_example = build_nn1(X_train_nn.shape[1])

nn1_example.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        37,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,953 (179.50 KB)

 Trainable params: 45,953 (179.50 KB)

 Non-trainable params: 0 (0.00 B)

### 4.3.2 Manual 5-Fold Cross-Validation — Architecture 1

Architecture 1 is evaluated using manual 5-fold cross-validation.
For each fold, a new neural network is created and trained on four
folds, while the remaining fold is used for validation.

MAE, MAPE, and R² are calculated for each validation fold and averaged
to evaluate the architecture's cross-validation performance.
Early stopping is used to prevent unnecessary training when validation
performance stops improving.

In [49]:
# Store validation metrics for each fold

nn1_fold_mae = []
nn1_fold_mape = []
nn1_fold_r2 = []

# Manual 5-fold cross-validation

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_nn), start=1):
    
    print(f"\nFold {fold} of 5")
    
    # Split the current fold into training and validation data
    
    X_fold_train = X_train_nn[train_idx]
    X_fold_val = X_train_nn[val_idx]
    
    y_fold_train = y_train_nn[train_idx]
    y_fold_val = y_train_nn[val_idx]
    
    # Create a fresh neural network for this fold
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)
    
    model = build_nn1(X_train_nn.shape[1])
    
    # Stop training when validation loss stops improving
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
    
    # Train the model
    
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Predict vehicle prices for the validation fold
    
    val_pred = model.predict(
        X_fold_val,
        verbose=0
    ).ravel()
    
    # Calculate validation metrics
    
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)
    
    # Store fold results
    
    nn1_fold_mae.append(fold_mae)
    nn1_fold_mape.append(fold_mape)
    nn1_fold_r2.append(fold_r2)
    
    # Display fold results
    
    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")


Fold 1 of 5

Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 17323.5137 - mae: 17323.5137 - mape: 57.5810 - val_loss: 6429.7085 - val_mae: 6429.7085 - val_mape: 23.5778
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 5412.4609 - mae: 5412.4609 - mape: 19.5243 - val_loss: 4712.3120 - val_mae: 4712.3120 - val_mape: 16.8271
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 4400.6313 - mae: 4400.6313 - mape: 15.2517 - val_loss: 4098.9058 - val_mae: 4098.9058 - val_mape: 14.1737
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3945.5757 - mae: 3945.5757 - mape: 13.3236 - val_loss: 3772.8169 - val_mae: 3772.8169 - val_mape: 12.8574
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3685.4871 - mae: 3685.4871 - mape: 12.3500 - val_loss: 3572.1614 - val_mae: 3572.1614 - val_mape: 12.1457
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3512.8621 - mae: 3512.8621 - mape: 11.7474 - val_loss: 3431.9546 - val_mae: 3431.9546 - val_ma

### 4.3.3 Architecture 1 Cross-Validation Results

The MAE, MAPE, and R² scores from the five validation folds are averaged
to summarize the overall cross-validation performance of Architecture 1.

In [50]:
nn1_cv_mae = np.mean(nn1_fold_mae)
nn1_cv_mape = np.mean(nn1_fold_mape)
nn1_cv_r2 = np.mean(nn1_fold_r2)

print(f"Architecture 1 Mean CV MAE: ${nn1_cv_mae:,.2f}")
print(f"Architecture 1 Mean CV MAPE: {nn1_cv_mape:.2%}")
print(f"Architecture 1 Mean CV R²: {nn1_cv_r2:.4f}")

Architecture 1 Mean CV MAE: $2,453.37
Architecture 1 Mean CV MAPE: 8.55%
Architecture 1 Mean CV R²: 0.8002


**Interpretation:**  
Keras Architecture 1 achieved a mean cross-validation MAE of $2,453.37,
a MAPE of 8.55%, and an R² of 0.8002. The neural network substantially
outperformed both Linear Regression and Ridge Regression, suggesting
that nonlinear relationships are important for predicting vehicle price.

However, Random Forest continues to provide the strongest
cross-validation performance, with lower MAE and MAPE and a higher R².
Additional neural network architectures will be evaluated to determine
whether changes in network structure improve performance.

### 4.4 Keras Architecture 2

Architecture 2 increases the depth and capacity of the neural network
by using three hidden Dense layers. The larger architecture is evaluated
to determine whether additional model complexity improves vehicle price
prediction.

In [51]:
def build_nn2(input_dim):
    
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )
    
    return model

In [52]:
# inspect to make sure it builds correctly

nn2_example = build_nn2(X_train_nn.shape[1])

nn2_example.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │        75,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,481 (455.00 KB)

 Trainable params: 116,481 (455.00 KB)

 Non-trainable params: 0 (0.00 B)

### 4.4.2 Manual 5-Fold Cross-Validation — Architecture 2

Architecture 2 is evaluated using the same manual 5-fold
cross-validation procedure used for Architecture 1. A new neural
network is created for each fold and evaluated using MAE, MAPE,
and R².

The same training settings are maintained so that differences in
performance can be attributed primarily to the change in neural
network architecture.

In [53]:
# Store validation metrics for each fold

nn2_fold_mae = []
nn2_fold_mape = []
nn2_fold_r2 = []

# Manual 5-fold cross-validation

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_nn), start=1):
    
    print(f"\nFold {fold} of 5")
    
    # Split the current fold into training and validation data
    
    X_fold_train = X_train_nn[train_idx]
    X_fold_val = X_train_nn[val_idx]
    
    y_fold_train = y_train_nn[train_idx]
    y_fold_val = y_train_nn[val_idx]
    
    # Create a fresh neural network for this fold
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)
    
    model = build_nn2(X_train_nn.shape[1])
    
    # Stop training when validation loss stops improving
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
    
    # Train the model
    
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Predict vehicle prices for the validation fold
    
    val_pred = model.predict(
        X_fold_val,
        verbose=0
    ).ravel()
    
    # Calculate validation metrics
    
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)
    
    # Store fold results
    
    nn2_fold_mae.append(fold_mae)
    nn2_fold_mape.append(fold_mape)
    nn2_fold_r2.append(fold_r2)
    
    # Display fold results
    
    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")


Fold 1 of 5
Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 8207.0586 - mae: 8207.0586 - mape: 28.1574 - val_loss: 3700.8489 - val_mae: 3700.8489 - val_mape: 12.4669
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3431.0435 - mae: 3431.0435 - mape: 11.4354 - val_loss: 3217.9226 - val_mae: 3217.9226 - val_mape: 10.8199
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3115.1235 - mae: 3115.1235 - mape: 10.3229 - val_loss: 3019.8293 - val_mae: 3019.8293 - val_mape: 10.2032
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2946.7961 - mae: 2946.7961 - mape: 9.7441 - val_loss: 2894.1038 - val_mae: 2894.1038 - val_mape: 9.8424
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 2828.3755 - mae: 2828.3755 - mape: 9.3464 - val_loss: 2806.8484 - val_mae: 2806.8484 - val_mape: 9.5788
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2736.0461 - mae: 2736.0461 - mape: 9.0402 - val_loss: 2735.0215 - val_mae: 2735.0215 - val_mape: 9.43

### 4.4.3 Architecture 2 Cross-Validation Results

The MAE, MAPE, and R² scores from the five validation folds are averaged
to summarize the cross-validation performance of Architecture 2.

In [63]:
nn2_cv_mae = np.mean(nn2_fold_mae)
nn2_cv_mape = np.mean(nn2_fold_mape)
nn2_cv_r2 = np.mean(nn2_fold_r2)

print(f"Architecture 2 Mean CV MAE: ${nn2_cv_mae:,.2f}")
print(f"Architecture 2 Mean CV MAPE: {nn2_cv_mape:.2%}")
print(f"Architecture 2 Mean CV R²: {nn2_cv_r2:.4f}")

Architecture 2 Mean CV MAE: $2,170.72
Architecture 2 Mean CV MAPE: 8.12%
Architecture 2 Mean CV R²: 0.8557


**Interpretation:**  
Architecture 2 achieved a mean cross-validation MAE of $2,170.72,
a MAPE of 8.12%, and an R² of 0.8557. It outperformed Architecture 1
across all three evaluation metrics.

The improvement suggests that increasing the depth and capacity of
the neural network allowed the model to capture additional patterns
in the vehicle data. However, Random Forest continues to provide
stronger cross-validation performance.

### 4.5 Keras Architecture 3

Architecture 3 introduces Dropout regularization between the hidden
Dense layers. Dropout randomly deactivates a portion of neurons during
training, which can reduce reliance on individual neurons and help
prevent overfitting.

This architecture is evaluated to determine whether adding
regularization improves cross-validation performance compared with
the previous neural network architectures.

In [55]:
def build_nn3(input_dim):
    
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)
    ])
    
    model.compile(
        optimizer="adam",
        loss="mae",
        metrics=["mae", "mape"]
    )
    
    return model

In [56]:
nn3_example = build_nn3(X_train_nn.shape[1])

nn3_example.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 256)            │        75,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,481 (455.00 KB)

 Trainable params: 116,481 (455.00 KB)

 Non-trainable params: 0 (0.00 B)

### 4.5.2 Manual 5-Fold Cross-Validation — Architecture 3

Architecture 3 is evaluated using the same manual 5-fold
cross-validation procedure used for the previous neural networks.

The same folds, optimizer, loss function, maximum number of epochs,
batch size, early stopping criteria, and evaluation metrics are used
to provide a consistent comparison among the three architectures.

In [57]:
# Store validation metrics for each fold

nn3_fold_mae = []
nn3_fold_mape = []
nn3_fold_r2 = []

# Manual 5-fold cross-validation

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_nn), start=1):
    
    print(f"\nFold {fold} of 5")
    
    # Split the current fold into training and validation data
    
    X_fold_train = X_train_nn[train_idx]
    X_fold_val = X_train_nn[val_idx]
    
    y_fold_train = y_train_nn[train_idx]
    y_fold_val = y_train_nn[val_idx]
    
    # Create a fresh neural network for this fold
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(42 + fold)
    
    model = build_nn3(X_train_nn.shape[1])
    
    # Stop training when validation loss stops improving
    
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
    
    # Train the model
    
    model.fit(
        X_fold_train,
        y_fold_train,
        validation_data=(X_fold_val, y_fold_val),
        epochs=50,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Predict vehicle prices for the validation fold
    
    val_pred = model.predict(
        X_fold_val,
        verbose=0
    ).ravel()
    
    # Calculate validation metrics
    
    fold_mae = mean_absolute_error(y_fold_val, val_pred)
    fold_mape = mean_absolute_percentage_error(y_fold_val, val_pred)
    fold_r2 = r2_score(y_fold_val, val_pred)
    
    # Store fold results
    
    nn3_fold_mae.append(fold_mae)
    nn3_fold_mape.append(fold_mape)
    nn3_fold_r2.append(fold_r2)
    
    # Display fold results
    
    print(f"Fold {fold} MAE: ${fold_mae:,.2f}")
    print(f"Fold {fold} MAPE: {fold_mape:.2%}")
    print(f"Fold {fold} R²: {fold_r2:.4f}")


Fold 1 of 5
Epoch 1/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 8546.9678 - mae: 8546.9678 - mape: 29.3406 - val_loss: 3815.5967 - val_mae: 3815.5967 - val_mape: 13.1028
Epoch 2/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 3966.6177 - mae: 3966.6177 - mape: 13.3912 - val_loss: 3330.1711 - val_mae: 3330.1711 - val_mape: 11.2649
Epoch 3/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 3680.0532 - mae: 3680.0532 - mape: 12.3873 - val_loss: 3172.1750 - val_mae: 3172.1750 - val_mape: 10.6145
Epoch 4/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 3556.5896 - mae: 3556.5896 - mape: 11.9715 - val_loss: 3126.1641 - val_mae: 3126.1641 - val_mape: 10.3880
Epoch 5/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 3477.6914 - mae: 3477.6914 - mape: 11.6736 - val_loss: 2946.2817 - val_mae: 2946.2817 - val_mape: 10.0133
Epoch 6/50
742/742 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 3408.7517 - mae: 3408.7517 - mape: 11.4570 - val_loss: 2915.0415 - val_mae: 2915.0415 - val_mape:

### 4.5.3 Architecture 3 Cross-Validation Results

The MAE, MAPE, and R² scores from the five validation folds are averaged
to summarize the cross-validation performance of Architecture 3.

In [58]:
nn3_cv_mae = np.mean(nn3_fold_mae)
nn3_cv_mape = np.mean(nn3_fold_mape)
nn3_cv_r2 = np.mean(nn3_fold_r2)

print(f"Architecture 3 Mean CV MAE: ${nn3_cv_mae:,.2f}")
print(f"Architecture 3 Mean CV MAPE: {nn3_cv_mape:.2%}")
print(f"Architecture 3 Mean CV R²: {nn3_cv_r2:.4f}")

Architecture 3 Mean CV MAE: $2,305.10
Architecture 3 Mean CV MAPE: 8.52%
Architecture 3 Mean CV R²: 0.8439


**Interpretation:**  
Architecture 3 achieved a mean cross-validation MAE of $2,305.10,
a MAPE of 8.52%, and an R² of 0.8439. Although Architecture 3
outperformed Architecture 1 in MAE and R², it did not outperform
Architecture 2.

Architecture 2 achieved the lowest MAE and MAPE and the highest R²
among the three neural network architectures. Therefore, Architecture 2
was selected as the final Keras model for retraining on the complete
training dataset.

## 4.6 Compare Keras Architectures

The mean cross-validation results from the three Keras architectures
are compared using MAE, MAPE, and R². The architecture with the lowest
mean cross-validation MAE is selected as the final Keras model.

Only the selected architecture will be retrained on the complete
training dataset and evaluated on the held-out test set.

In [59]:
keras_cv_results = pd.DataFrame({
    "Architecture": [
        "Architecture 1",
        "Architecture 2",
        "Architecture 3"
    ],
    "CV MAE": [
        nn1_cv_mae,
        nn2_cv_mae,
        nn3_cv_mae
    ],
    "CV MAPE": [
        nn1_cv_mape,
        nn2_cv_mape,
        nn3_cv_mape
    ],
    "CV R2": [
        nn1_cv_r2,
        nn2_cv_r2,
        nn3_cv_r2
    ]
})

keras_cv_results

,Architecture,CV MAE,CV MAPE,CV R2
0,Architecture 1,2453.369434,0.085489,0.800237
1,Architecture 2,2170.721240,0.081171,0.855730
2,Architecture 3,2305.104980,0.085165,0.843881


### 4.6.2 Select the Best Keras Architecture

The architecture with the lowest mean cross-validation MAE is selected
as the winning neural network. MAPE and R² are also reviewed to provide
additional context about model performance.

In [60]:
best_keras_row = keras_cv_results.loc[
    keras_cv_results["CV MAE"].idxmin()
]

print("Best Keras Architecture:")
print(best_keras_row)

Best Keras Architecture:
Architecture    Architecture 2
CV MAE              2170.72124
CV MAPE               0.081171
CV R2                  0.85573
Name: 1, dtype: object


## 4.7 Retrain the Winning Keras Architecture

Architecture 2 produced the best cross-validation performance and is
therefore selected as the final Keras model.

A new Architecture 2 model is created and retrained using the complete
training dataset. The training time for this final refit is recorded
for comparison with the scikit-learn models.

In [67]:
# Clear the interrupted/previous Keras model from memory
tf.keras.backend.clear_session()
tf.random.set_seed(42)

# Build a fresh copy of the winning architecture
best_keras = build_nn2(X_train_nn.shape[1])

# Time the final refit on the complete training set
start_time = time.perf_counter()

history = best_keras.fit(
    X_train_nn,
    y_train_nn,
    epochs=50,
    batch_size=256,
    verbose=1
)

keras_train_time = time.perf_counter() - start_time

print(f"Final Keras training time: {keras_train_time:.2f} seconds")

Epoch 1/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 7378.1670 - mae: 7378.1670 - mape: 25.2121
Epoch 2/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3310.0847 - mae: 3310.0847 - mape: 11.0389
Epoch 3/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3021.5132 - mae: 3021.5132 - mape: 10.0497
Epoch 4/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2867.4880 - mae: 2867.4880 - mape: 9.5226
Epoch 5/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2759.1777 - mae: 2759.1777 - mape: 9.1666
Epoch 6/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2669.7617 - mae: 2669.7617 - mape: 8.8752
Epoch 7/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2590.2446 - mae: 2590.2446 - mape: 8.6262
Epoch 8/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2518.0330 - mae: 2518.0330 - mape: 8.4111
Epoch 9/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2448.7104 - mae: 2448.7104 - mape: 8.2097
Epoch 10/50
927/927 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2386.0298 - mae: 2

### 4.8 Evaluate Winning Keras Model on Held-Out Test Set

The final Architecture 2 model is evaluated on the held-out test set, which
was not used during model training or cross-validation. Performance is
measured using MAE, MAPE, and R² so that the final Keras model can be
compared with the scikit-learn regression models.

In [69]:
# Generate predictions on the held-out test set
keras_test_pred = best_keras.predict(
    X_test_nn,
    verbose=0
).ravel()

# Calculate test-set performance metrics
keras_test_mae = mean_absolute_error(
    y_test_nn,
    keras_test_pred
)

keras_test_mape = mean_absolute_percentage_error(
    y_test_nn,
    keras_test_pred
)

keras_test_r2 = r2_score(
    y_test_nn,
    keras_test_pred
)

# Display results
print(f"Keras Architecture 2 Test MAE: ${keras_test_mae:,.2f}")
print(f"Keras Architecture 2 Test MAPE: {keras_test_mape:.2%}")
print(f"Keras Architecture 2 Test R²: {keras_test_r2:.4f}")

Keras Architecture 2 Test MAE: $2,039.40
Keras Architecture 2 Test MAPE: 7.72%
Keras Architecture 2 Test R²: 0.9263


### 4.9 Record and Save Winning Keras Model

The final performance metrics for Architecture 2 are added to the model
results. The trained Architecture 2 model is then saved for later comparison
and use.

In [70]:
# Add the winning Keras model to the results
model_results.append({
    "Model": "Keras Architecture 2",
    "Best Parameters": "256-128-64, ReLU, Adam",
    "CV MAE": nn2_cv_mae,
    "CV MAPE": nn2_cv_mape,
    "CV R2": nn2_cv_r2,
    "Test MAE": keras_test_mae,
    "Test MAPE": keras_test_mape,
    "Test R2": keras_test_r2,
    "Training Time (sec)": keras_train_time
})

# Display the updated results
results_df = pd.DataFrame(model_results)
display(results_df)

,Model,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
0,Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,78.251818
1,Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
2,Ridge Regression,{'alpha': 100.0},5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
3,Keras Architecture 2,"256-128-64, ReLU, Adam",2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758


In [72]:
results_df = (
    pd.DataFrame(model_results)
    .drop_duplicates(subset="Model", keep="last")
    .set_index("Model")
)

display(results_df)

,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,,
Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,2.370951
Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
Ridge Regression,{'alpha': 100.0},5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
Keras Architecture 2,"256-128-64, ReLU, Adam",2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758


In [73]:
best_keras.save("../../models/P1_N3_nn2.keras")

print("Winning Keras model saved successfully.")

Winning Keras model saved successfully.


In [76]:
display(results_df)

print("\nTable shape:", results_df.shape)
print("\nIndex:")
print(results_df.index.tolist())

print("\nMissing values:", results_df.isna().sum().sum())
print("Duplicate models:", results_df.index.duplicated().sum())

,Best Parameters,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,,
Linear Regression,"{'fit_intercept': True, 'positive': False}",5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,2.370951
Random Forest,"{'max_depth': None, 'max_features': 0.5, 'min_...",1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
Ridge Regression,{'alpha': 100.0},5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
Keras Architecture 2,"256-128-64, ReLU, Adam",2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758



Table shape: (4, 8)

Index:
['Linear Regression', 'Random Forest', 'Ridge Regression', 'Keras Architecture 2']

Missing values: 0
Duplicate models: 0


In [80]:
# Create the assignment-format performance table for export
model_metrics = results_df[
    [
        "CV MAE",
        "CV MAPE",
        "CV R2",
        "Test MAE",
        "Test MAPE",
        "Test R2",
        "Training Time (sec)"
    ]
].copy()

display(model_metrics)

# Export for use in Notebook 4
metrics_path = "../../data/model_metrics.csv"
model_metrics.to_csv(metrics_path, index=True)

,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,2.370951
Random Forest,1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
Ridge Regression,5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
Keras Architecture 2,2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758


In [81]:
from pathlib import Path

files_to_check = {
    "Model metrics":
        "../../data/model_metrics.csv",
    "Linear Regression":
        "../../models/P1_N3_linreg.pkl",
    "Random Forest":
        "../../models/P1_N3_rf.pkl",
    "Ridge Regression":
        "../../models/P1_N3_ridge.pkl",
    "Keras Architecture 2":
        "../../models/P1_N3_nn2.keras"
}

print("FINAL NOTEBOOK 3 EXPORT CHECK")
print("-" * 40)

for name, path in files_to_check.items():
    print(f"{name:<25} {Path(path).exists()}")

print("\nExported metrics shape:")
check_metrics = pd.read_csv(
    "../../data/model_metrics.csv",
    index_col="Model"
)

print(check_metrics.shape)
display(check_metrics)

FINAL NOTEBOOK 3 EXPORT CHECK
----------------------------------------
Model metrics             True
Linear Regression         True
Random Forest             True
Ridge Regression          True
Keras Architecture 2      True

Exported metrics shape:
(4, 7)


,CV MAE,CV MAPE,CV R2,Test MAE,Test MAPE,Test R2,Training Time (sec)
Model,,,,,,,
Linear Regression,5554.992861,0.267481,0.680132,5498.223400,0.268621,0.733916,2.370951
Random Forest,1739.876978,0.071811,0.897929,1638.968606,0.069163,0.950291,122.248554
Ridge Regression,5550.671509,0.267161,0.680153,5494.759379,0.268372,0.733965,2.263217
Keras Architecture 2,2170.721240,0.081171,0.855730,2039.402222,0.077246,0.926307,122.726758


## 5. Model Performance Discussion

Four final regression models were evaluated: Linear Regression, Random Forest,
Ridge Regression, and the winning Keras neural network (Architecture 2).
Performance was compared using mean cross-validation MAE, MAPE, and R², as
well as the same metrics on the held-out test set.

### Overall Model Comparison

Random Forest produced the strongest overall performance. It achieved a mean
cross-validation MAE of $1,739.88 and a test MAE of $1,638.97. Its test MAPE
was 6.92%, meaning its predictions differed from actual vehicle prices by
approximately 6.92% on average. Random Forest also achieved the highest test
R² of 0.9503, indicating that it explained approximately 95.03% of the
variation in vehicle prices in the held-out test set.

Keras Architecture 2 was the second-best model. It achieved a mean
cross-validation MAE of $2,170.72 and a test MAE of $2,039.40. Its test MAPE
was 7.72%, and its test R² was 0.9263. The neural network therefore performed
substantially better than the linear models, although it did not outperform
Random Forest.

Linear Regression and Ridge Regression produced very similar results. Linear
Regression had a test MAE of $5,498.22 and test R² of 0.7339, while Ridge
Regression had a test MAE of $5,494.76 and test R² of 0.7340. Ridge
regularization produced only a very small improvement over ordinary Linear
Regression. This suggests that regularization alone did not address the main
limitations of the linear models. Of the four final models, Linear Regression 
had the highest test MAE and was therefore the worst-performing model overall, 
although its performance was nearly identical to Ridge Regression.

### Why Random Forest Performed Best

The strong performance of Random Forest suggests that relationships between
vehicle characteristics and price are not entirely linear. Vehicle price appears 
to depend on complex interactions among characteristics such as horsepower,
mileage, vehicle age, make, model, body type, and other features. Random
Forest can model nonlinear relationships and interactions without requiring
them to be specified manually, which likely contributed to its stronger
performance compared with Linear and Ridge Regression.

The Keras neural network can also learn nonlinear relationships, which is
consistent with its strong performance relative to the linear models.
However, for this dataset, Random Forest captured the predictive patterns
more effectively than the tested neural-network architectures.

### Cross-Validation and Test Performance

For both Random Forest and Keras Architecture 2, performance on the held-out
test set was somewhat better than the average cross-validation performance.
The test set may have been somewhat easier to predict than some of the
cross-validation folds. In particular, the training data contained more
extreme vehicle prices than the held-out test data, which can affect
regression metrics such as MAE and R².

### R² and Leakage Check

Random Forest achieved a high test R² of 0.9503. Because unusually high R²
values can sometimes indicate target leakage, the modeling data were checked
before training. The target variable, price, was excluded from the predictor
matrices, object-dtype metadata columns were removed from the modeling
features, and the original training/test separation from preprocessing was
preserved. Therefore, no direct price leakage was identified during these
checks. The high Random Forest R² should still be interpreted together with
the cross-validation results rather than relying on the test R² alone.

### Final Model Selection

Based on the combined cross-validation and held-out test results, Random
Forest was the best-performing model in Notebook 3. It produced the lowest
MAE and MAPE and the highest R² among the four final models. Keras
Architecture 2 ranked second, while Linear Regression and Ridge Regression
showed substantially larger prediction errors.